### Функции для работы с MySQL

In [1]:
import os
from dotenv import load_dotenv
import pymysql

load_dotenv()

def mysqlconn():
    """
    Установка подключения к базе данных sakila в MySQL
    """
    try:
        connection = pymysql.connect(
            host = os.getenv('DB_HOST'),
            user = os.getenv('DB_USER'),
            password = os.getenv('DB_PASSWORD'),
            database = os.getenv('DB_NAME'),
            port = 3306
        )
        # print('Подключение к MySQL установлено')
        return connection
    except pymysql.MySQLError as e:
        print(f"Ошибка подключения: {e}")


# Забираем информацию по столбцам: id, название, жанр, год, рейтинг
def get_films_info() -> dict[str,int]: 
    """
    Загрузка списка фильмов из БД в словарь. f.film_id - ключ, 
    значение - словарь с данными f.title, c.name, f.release_year, f.rating
    """
    dict_film = {} #Пустой словарик пополняем из курсора (кроме film_id)
    conn1 = mysqlconn()
    with conn1.cursor() as cursor:
        cursor.execute("""
            SELECT 
                f.film_id, f.title, c.name, f.release_year, f.rating  
            FROM 
                sakila.film as f
            LEFT JOIN 
                film_category as fc ON f.film_id = fc.film_id
            LEFT JOIN 
                category as c ON fc.category_id = c.category_id
            """ )
        for film in cursor:
            # print(film)
            dict_film[film[0]] = {
                "title": film[1], 
                "genre": film[2], 
                "year": film[3], 
                "rating": film[4]
            }
    return dict_film


# Джоиним таблицы "category" и "film_category" 
# и забираем название жанров и агрегированный min и max релиза
def get_genres_info() -> list[tuple[str, int, int]]:
    """
    Получение списка жанров и их временного диапазона 
    """
    conn2 = mysqlconn()
    genres_list = []
    with conn2.cursor() as cursor:
        cursor.execute("""
            SELECT 
                c.name, MIN(f.release_year), MAX(f.release_year)
            FROM 
                category as c
            JOIN 
                film_category as fc ON c.category_id = fc.category_id
            JOIN 
                film as f ON fc.film_id = f.film_id
            GROUP BY c.name
            ORDER BY c.name
            """)
        for row in cursor:
            # print(row)
            genres_list.append(row)
        return genres_list

In [2]:
# get_films_info()
# get_genres_info()

### Функции для работы с MongoDB

In [3]:
import pymongo
import datetime


def mongoconn():
    """
    Установка связи с MongoDB
    """
    global client
    client = pymongo.MongoClient(os.getenv('MONGO_URL'))
    py_client = client
    db = client['ich_edit']
    collection = db['final_project_120925_Schaefer']
    return collection


def log_write(search_type: str, params:dict[str, int], results_count:int):
    """
    Запись результатов поиска в MongoDB
    """
    log_line = {
        'timestamp': datetime.datetime.now(),
        'search_type': search_type,
        'params': params,
        'results_count': results_count
    }
    try:
        coll = mongoconn()
        coll.insert_one(log_line)
        # print(f'Лог записан: {log_line}')
    except Exception as e:
        print(f"Ошибка записи в MongoDB: {e}")


def get_log_keywords() -> list[dict[str, int]]:
    """
    Получение топ-5 логов по ключевым словам
    """    
    coll2 = mongoconn()
    ret_keywords = coll2.aggregate([
        {'$match': {'search_type': 'keyword'}},
        {'$group': {'_id': '$params.keyword', 'count': {'$sum': 1}}},
        {'$sort': {'count': -1}},
        {'$limit': 5}
    ])
    return ret_keywords


def get_log_genres() -> list[dict[str, int]]:
    """
    Получение топ-5 логов по жанрам-годам
    """    
    coll3 = mongoconn()
    ret_genres = coll3.aggregate([
        {'$match': {'search_type': 'genre_year'}},
        {'$group': {'_id': {
            'genre': '$params.genre',
            'start_year': '$params.start_year',
            'end_year': '$params.end_year'},
        'total': {'$sum': 1}}},
        {'$sort': {'total': -1}}, 
        {'$limit': 5}
    ])
    return ret_genres

    
def get_log_last5() -> list[dict[str, int]]:
    """
    Получение топ-5 логов по последней дате записи
    """    
    coll4 = mongoconn()
    ret_last5 = coll4.aggregate([
        {'$sort': {'timestamp': -1}},
        {"$group": 
             {"_id": {
                "search_type": "$search_type",
                "params": "$params"},
            "total_count": {"$sum": 1},
            "last_timestamp": {"$max": "$timestamp"},
            "results_count": {"$first": "$results_count"}}},
        {"$sort": {"last_timestamp": -1}},
        {'$limit': 5}
    ])
    return ret_last5


def show_statistics():
    """
    Меню статистики и вывод поисковых результатов из MongoDB
    """        
    while True:
        print("\n" + "="*3 + " Статистика запросов (MongoDB) " + "="*45)
        print("1 - TOP-5 популярных запросов по {keyword}")
        print("2 - TOP-5 популярных запросов по {genres-years}")
        print("3 - TOP-5 последних запросов")
        print("4 - Выход в главное меню")
        print("="*79)

        stat_choice = input("Выберите пункт меню для вывода статистики: ").strip()
        if stat_choice not in ['1', '2', '3', '4']:
            print("Пожалуйста, введите корректное число от 1 до 4.")
            continue
        if stat_choice == '1':
            stats = get_log_keywords()
            print()
            print("TOP-5 популярных запросов по {keyword}:")
            print('-'*79)
            print(f"| {'#':<4} | {'Ключевое слово':<43} | "
                  f"{'Кол-во запросов':<22} | ")
            print('-'*79)
            for i, item in enumerate(stats, 1):
                # keyword = item.get('params', {})
                print(f"| {i:<4} | {item.get('_id', 'N/A'):<43} | "
                      f"{item.get('count', 0):<22} | ")
            print('-'*79)

        elif stat_choice == '2':
            stats = get_log_genres()
            print()
            print("TOP-5 популярных запросов по {genres-years}:")
            print('-'*79)
            print(f"| {'#':<4} | {'Жанр':<21} | {'Годы':<20} | "
                  f"{'Кол-во запросов':<21} |")
            print('-'*79)
            for i, item in enumerate(stats, 1):
                p = item.get('_id', {})
                genre_range = f"{p.get('start_year')}-{p.get('end_year')}"
                print(f"| {i:<4} | {p.get('genre', 'N/A'):<21} | "
                      f"{genre_range:<20} | {item.get('total', 0):<21} |")
            print('-'*79)

        elif stat_choice == '3':
            last_searches = get_log_last5()
            print()
            print("TOP-5 последних запросов:")
            print('-'*94)
            print(f"| {'#':<3} | {'Запрос':<10} | {'Что искали':<21} | "
                  f"{'Кол-во':<8} | {'Найдено':<7} | {'Дата - время':<26} |")
            print(f"| {'':<3} | {'':<10} | {'':<21} | "
                  f"{'запросов':<8} | {'фильмов':<7} | "
                  f"{'последнего запроса':<26} |")
            print('-'*94)
            for i, item in enumerate(last_searches, 1):
                p = item.get('_id', {}).get('params', {})
                s = item.get('_id', {}).get('search_type', 'N/A')
                if s == 'keyword':
                    desc = p.get('keyword', 'N/A')
                else:
                    desc = (
                        f"{p.get('genre', 'N/A')} "
                        f"{p.get('start_year')}-{p.get('end_year')}"
                    )
                print(f"| {i:<3} | {s:<10} | {desc:<21} | "
                      f"{str(item.get('total_count', 0)):<8} | "
                      f"{str(item.get('results_count', 0)):<7} | "
                      f"{str(item.get('last_timestamp', 'N/A')):<26} |"
                )
            print('-'*94)
            
        elif stat_choice == '4':
            break

### Функции поиска  

In [4]:
def search_key_word():
    """
    Запрашивает ключевое слово для поиска
    и ищет совпадения в названиях фильмов

    Возвращает кортеж из results, key_word, где:
    results - список словатей с найденными результатами
    title, genre, year, rating
    key_word - ключевое слово, которое было использовано для поиска
    
    """
    results = []
    key_word = input(f"Введите ключевое слово "
                     f"или название фильма:").strip().lower()

        # Поиск в списке фильмов
    for num, result in get_films_info().items():
        # Проверка вхождения слова
        if key_word in result['title'].lower(): 
            results.append(result)
    print(f"По слову '{key_word}' найдено фильмов: {len(results)}")
    return results, key_word


def search_genre_year():
    """
    Выводит перечень жанров, запрашивает выбор жанра и диапазона лет,
    выполняет поиск в базе данных

    Возвращает кортеж из target_genre, year_start, year_end, results
    """
    genres_data = get_genres_info()  
                        
    # Вывод список жанров для выбора
    print("\nДоступные жанры:")
    print('-'*36)
    print(f"| {'#':<3} | {'Жанр':<12} | {'Период':<11} |")
    print('-'*36)            
    for i, g in enumerate(genres_data, 1):
        print(f"| {i:<3} | {g[0]:<12} | ({g[1]}-{g[2]}) |")
    print('-'*36)

    while True:
        try:
            print()
            g_choice = int(input("Введите номер жанра (или 0 для выхода): "))
            if g_choice == 0:
                return None
            if 1 <= g_choice <= len(genres_data):
                target_genre = genres_data[g_choice - 1][0]
                print(f"Для жанра '{genres_data[g_choice - 1][0]}'"
                     f" доступен диапазон с {genres_data[g_choice - 1][1]}"
                     f" по {genres_data[g_choice - 1][2]} год")
                year_start = int(input("Введите год начала: "))
                year_end = int(input("Введите год конца: "))
                if year_start > year_end:
                    print("Ошибка: Год начала не может быть больше года конца.")
                    continue
                if not (int(genres_data[g_choice - 1][1]) <= year_start
                        <= int(genres_data[g_choice - 1][2]) and 
                        int(genres_data[g_choice - 1][2]) >= year_end 
                        >= int(genres_data[g_choice - 1][1])):
                    print(f"Ошибка: Годы должны быть в диапазоне от "
                        f"{genres_data[g_choice - 1][1]} до "
                        f"{genres_data[g_choice - 1][2]}.")
                    continue
                    
                # Прямой поиск фильмов в БД
                results = []
                conn = mysqlconn()
                with conn.cursor() as cursor:
                    cursor.execute(
                        """
                        SELECT f.title, f.release_year, f.rating
                        FROM film f
                        JOIN film_category fc ON f.film_id = fc.film_id
                        JOIN category c ON fc.category_id = c.category_id
                        WHERE c.name = %s 
                        AND f.release_year BETWEEN %s AND %s
                    """
                    , (target_genre, year_start, year_end))
    
                    for row in cursor:
                        # print(row)
                        results.append(row)
                print()
                print(f"В жанре '{genres_data[g_choice - 1][0]}' за период с "
                      f"{year_start} года по { year_end} год найдено фильмов: "
                      f"{len(results)}")
                return target_genre, year_start, year_end, results
            else:
                print("Неверный номер жанра.")
        except ValueError:
            print("Ошибка: Пожалуйста, введите корректное число.")
                    
    

In [5]:
# search_key_word()
# search_genre_year()

### Функция вывода с пагинацией

In [6]:
def main():
    """
    Функция с меню интерфейса, поиска по фильмам, жанрам
    и просмотра статистики
    
    - Поиск фильмов по ключевому слову в названии
    - Поиск фильмов по комбинации жанра и временного диапазона
    - Демонстрация статистики поисковых запросов

    Результаты отображаются группами по 10 записей с возможностью
    продолжения их просмотра или выхода в главное меню
    
    """
    connection = None
    while True:
        print("\n" + "="*3 + " Меню " + "="*70)
        print("1 - Поиск по названию фильма / {keyword}")
        print("2 - Поиск по жанру и диапазону годов / {genres-years}")
        print("3 - Статистика")
        print("4 - Выход")
        print("="*79)
        
        choice = input("Выбери пункт меню: ")
        if choice not in ['1', '2', '3', '4']:
            print("Пожалуйста, введите корректное число от 1 до 4.")
            continue

# ---------------Выбираем по ключевому слову-----------------------------------
        if choice == '1':
            results, key_word = search_key_word()

            #Логирование
            log_write("keyword", {'keyword': key_word}, len(results))
            
            # Пагинация            
            offset = 0
            limit = 10
            while offset < len(results):
                print('-'*79)
                print(f"| {'#':<2} | {'Название фильма':<38} | "
                      f"{'Жанр':<12} | {'Год':<4} | {'Рейтинг':<7} |")
                print('-'*79)
                for num, result in enumerate(
                    results[offset : offset + limit], start = offset + 1
                ):
                    # print(num, result)
                    print(f"| {num:<2} | {result['title']:<38} | "
                          f"{result['genre']:<12} | {result['year']:<4} | "
                          f"{result['rating']:<7} |")
                print('-'*79)
                print()
                if offset + limit >= len(results):
                    print("Вы просмотрели весь список.")
                    input("Нажмите Enter, чтобы вернуться в главное меню ")
                    break

                # Листаем дальше
                print(f"Вывести следующие 10?")
                q = input(f'Нажмите "Y" для продолжения или '
                          f'любую клавишу для выхода: '.strip().lower())
                if q == "y" or q == "у":
                    offset += limit
                else:
                    print("Поиск завершен")
                
                    break

# ---------------Выбираем по жанру---------------------------------------------        
        elif choice == '2':
            search_data = search_genre_year()
            if search_data is None:
                print("Поиск по жанрам отменен.")
                continue
            target_genre, year_start, year_end, results = search_data

            # Логирование
            log_params = {
                'genre': target_genre, 
                'start_year': year_start, 
                'end_year': year_end
            }
            log_write("genre_year", log_params, len(results))
            
            # Пагинация
            offset = 0
            limit = 10
            if len(results) == 0:
                input("Нажмите Enter для возврата в меню ")
            while offset < len(results):
                print('-'*79) 
                print(f"| {'#':<3} | {'Название фильма':<45} | "
                      f"{'Год выпуска'} | {'Рейтинг':<7} |")
                print('-'*79) 
                for num, genre in enumerate(
                    results[offset : offset + limit], start = offset + 1
                ):
                    print(f"| {num:<3} | {genre[0]:<45} | "
                        f"{genre[1]:<11} | {genre[2]:<7} |")
                print('-'*79) 
                print()
                if offset + limit >= len(results):
                    print("Вы просмотрели весь список.")
                    input("Нажмите Enter, чтобы вернуться в главное меню ")
                    break 
                
                # Листаем дальше
                print(f"Вывести следующие 10?")
                q = input(f'Нажмите "Y" для продолжения или '
                          f'любую клавишу для выхода: '.strip().lower())
                if q == "y" or q == "у":
                    offset += limit
                else:
                    print("Поиск завершен")
                    break 

        elif choice == '3':
            show_statistics()
            
        elif choice == '4':
            if connection:
                connection.close()
                py_client.close()
            print("Вы вышли из программы")
            print("Соединения с БД закрыты. Программа завершена.")
            break
            

### Запуск программы

In [7]:
main()


=== Меню ======================================================================
1 - Поиск по названию фильма / {keyword}
2 - Поиск по жанру и диапазону годов / {genres-years}
3 - Статистика
4 - Выход


Выбери пункт меню:  1
Введите ключевое слово или название фильма: еук


По слову 'еук' найдено фильмов: 0

=== Меню ======================================================================
1 - Поиск по названию фильма / {keyword}
2 - Поиск по жанру и диапазону годов / {genres-years}
3 - Статистика
4 - Выход


KeyboardInterrupt: Interrupted by user